## GPU Node Quick Start

This notebook adds a GPU worker node (node4) to your existing Kubernetes cluster.

**Prerequisites:**
- Your base 3-node cluster is already running (completed quick start or full setup)
- You have a GPU reservation on Chameleon Cloud
- Terraform, Ansible, and Kubespray are already set up

In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH
export PYTHONUSERBASE=/work/.local

export NET_ID=netID
export OS_AUTH_URL=https://kvm.tacc.chameleoncloud.org:5000/v3
export OS_PROJECT_NAME="CHI-XXXXXX"
export OS_REGION_NAME="KVM@TACC"

In [ ]:
# runs in Chameleon Jupyter environment
# Create a separate lease for the GPU node
openstack reservation lease create lease_gpu_$NET_ID \
  --start-date "$(date -u '+%Y-%m-%d %H:%M')" \
  --end-date "$(date -u -d '+4 hours' '+%Y-%m-%d %H:%M')" \
  --reservation "resource_type=flavor:instance,flavor_id=$(openstack flavor show g1.h100.pci.1 -f value -c id),amount=1"

In [ ]:
# runs in Chameleon Jupyter environment
export TF_VAR_suffix=$NET_ID
export TF_VAR_key=id_rsa_chameleon
export TF_VAR_reservation=$(openstack reservation lease show lease_mlops_$NET_ID -f json -c reservations \
      | jq -r '.reservations[0].flavor_id')
export TF_VAR_gpu_reservation=$(openstack reservation lease show lease_gpu_$NET_ID -f json -c reservations \
      | jq -r '.reservations[0].flavor_id')
echo "Base reservation: $TF_VAR_reservation"
echo "GPU reservation: $TF_VAR_gpu_reservation"

In [ ]:
# runs in Chameleon Jupyter environment
unset $(set | grep -o "^OS_[A-Za-z0-9_]*")
cd /work/gourmetgram-iac/tf/kvm
terraform apply -auto-approve

In [ ]:
# runs in Chameleon Jupyter environment
cd /work/gourmetgram-iac/ansible
ansible-playbook -i inventory.yml pre_k8s/pre_k8s_configure.yml --limit node4

In [ ]:
# runs in Chameleon Jupyter environment
export ANSIBLE_CONFIG=/work/gourmetgram-iac/ansible/ansible.cfg
export ANSIBLE_ROLES_PATH=roles
cd /work/gourmetgram-iac/ansible/k8s/kubespray
# Refresh facts for all nodes first, then add node4
ansible-playbook -i ../inventory/mycluster --become --become-user=root ./facts.yml
ansible-playbook -i ../inventory/mycluster --become --become-user=root ./scale.yml --limit=node4

In [ ]:
# runs in Chameleon Jupyter environment
cd /work/gourmetgram-iac/ansible
ansible-playbook -i inventory.yml post_k8s/post_k8s_gpu.yml

In [ ]:
# runs in Chameleon Jupyter environment
cd /work/gourmetgram-iac/ansible
ansible -i inventory.yml node1 -m copy -a "src=/work/gourmetgram-iac/workflows/train-model-gpu.yaml dest=/tmp/train-model-gpu.yaml" --become --become-user=cc
ansible -i inventory.yml node1 -m command -a "kubectl apply -n argo -f /tmp/train-model-gpu.yaml" --become --become-user=cc

In [ ]:
# runs in Chameleon Jupyter environment
cd /work/gourmetgram-iac/ansible
ansible -i inventory.yml node1 -m shell -a "kubectl get nodes -o wide && echo '---' && kubectl describe node node4 | grep -A10 'Capacity:' && echo '---' && kubectl describe node node4 | grep -A5 'Taints:'" --become --become-user=cc

In [ ]:
# runs in Chameleon Jupyter environment
# un-comment to remove GPU node
# cd /work/gourmetgram-iac/tf/kvm
# terraform destroy -target=openstack_compute_instance_v2.gpu_node -auto-approve